# Time & cost omnibus — Friedman

Do wall-clock time and USD cost differ across backends (omnibus)?

In [ ]:
from analysis import load_runs

# Pin matplotlib (Agg) + RNG seeds for reproducible, headless figures.
plt = load_runs.pin_style(seed=0)

# Load real runs if any exist, else the deterministic synthetic dataset.
records = load_runs.load_all()
backends = load_runs.backend_names(records)
STRATEGY = "zero_shot"  # the held-fixed strategy for the cross-backend tests
print(f"{len(records)} runs, backends={backends}")

In [ ]:
import numpy as np

from analysis import stats as st
from analysis.aggregate import aggregate_runs, build_metric_matrix

summaries = aggregate_runs(records)
omnibus = {}
for metric in ("wall_clock", "cost"):
    ids, mat = build_metric_matrix(summaries, metric=metric, backends=backends, strategy=STRATEGY)
    omnibus[metric] = st.friedman(mat)
    print(f"{metric}: {omnibus[metric]}")

In [ ]:
medians = {m: [] for m in ("wall_clock", "cost")}
field = {"wall_clock": "median_wall_clock_s", "cost": "median_usd_cost"}
for m in medians:
    for b in backends:
        vals = [
            getattr(s, field[m])
            for s in summaries
            if s.backend_name == b and s.strategy == STRATEGY
        ]
        medians[m].append(float(np.median(vals)) if vals else 0.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, m in zip(axes, ("wall_clock", "cost"), strict=True):
    ax.bar(backends, medians[m], color="indianred")
    ax.set_title(f"median {m} (p={omnibus[m].p_value:.3g})")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
axes[0].set_ylabel("seconds")
axes[1].set_ylabel("USD")
load_runs.save_figure(fig, "03_time_cost_omnibus", "median_time_cost")
fig

In [ ]:
from IPython.display import Markdown

rows = [
    f"- **{m}**: chi2 = {r.statistic:.3f}, df = {r.df}, p = {r.p_value:.4g}, "
    f"Kendall's W = {r.kendalls_w:.3f} "
    f"({'reject H0' if r.p_value < 0.05 else 'fail to reject H0'})"
    for m, r in omnibus.items()
]
Markdown("### Summary — time/cost omnibus (Friedman)\n\n" + "\n".join(rows))